# Notebook 17 — 3B Ceiling Baseline · PIQA
## SLM-to-SLM Guided Reasoning — Compute Ceiling Comparison

**Purpose:** Run Qwen2.5-3B alone with 5 votes — no fine-tuning, no LoRA, no guide.
This is the **compute ceiling** (15B param-passes) vs our pipeline's 10.5B.

| Condition | Setup | Compute |
|-----------|-------|---------|
| Baseline | 1.5B × 5 | 7.5B |
| **Our Pipeline** | 3B guide (LoRA) + 1.5B × 5 | **10.5B** |
| **← This notebook** | 3B base × 5 | **15.0B** |

Same 500 PIQA questions · Same seed=42 · Same temp=0.4 · Same 3-angle evaluation

**Why PIQA is the most important ceiling test:** The 1.5B baseline had a catastrophic A-option
collapse (93.4% of votes going to A). The pipeline's fine-tuned guide corrected this to 42.1%.
The key question here: does the 3B base model also suffer from A-bias, or does raw model
size suppress the positional heuristic on its own?

In [1]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print('HuggingFace login done')

HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU check
import os, json, re, time
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/piqa_ceiling"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"PyTorch : {torch.__version__}")
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {props.total_memory/1024**3:.1f} GB")
else:
    print("No GPU detected")
print(f"Output  : {OUTPUT_DIR}")

PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 15.9 GB
Output  : /kaggle/working/piqa_ceiling


In [4]:
# CELL 4 -- Configuration
# ONE model only. No guide. No LoRA. No fine-tuning.
# Ceiling condition: 3B x 5 votes = 15B param-passes.
CONFIG = {
    "model_name"          : "Qwen/Qwen2.5-3B-Instruct",
    "model_params_B"      : 3.0,
    # Dataset
    "dataset_name"        : "nthngdy/piqa",
    "dataset_split"       : "validation",     # test labels withheld; validation has labels
    "max_eval_samples"    : 500,              # same as pipeline N=500 run
    "random_seed"         : 42,              # FIXED -- must match pipeline run exactly
    # Voting
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,             # same as pipeline N=500 run
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 300,             # binary choice -- shorter answers
    # Random chance for PIQA (binary A/B)
    "random_chance"       : 50.0,
    # Known results from N=500 pipeline run (for comparison in Angle 1)
    "known_baseline_acc"  : 54.2,            # 1.5B x 5 @ 7.5B
    "known_pipeline_acc"  : 78.4,            # 3B+LoRA + 1.5B x 5 @ 10.5B
    "known_baseline_B"    : 7.5,
    "known_pipeline_B"    : 10.5,
    # Known position bias from N=500 pipeline run
    "known_pipeline_A_pct"  : 42.1,          # guided: A=42.1% B=57.9%
    "known_baseline_A_pct"  : 93.4,          # baseline: A=93.4% B=6.6% (catastrophic)
    # Output paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}
print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<28}: {v}")

Config ready:
  model_name                  : Qwen/Qwen2.5-3B-Instruct
  model_params_B              : 3.0
  dataset_name                : nthngdy/piqa
  dataset_split               : validation
  max_eval_samples            : 500
  random_seed                 : 42
  n_votes                     : 5
  vote_temperature            : 0.4
  refiner_temperature         : 0.3
  max_new_tokens              : 300
  random_chance               : 50.0
  known_baseline_acc          : 54.2
  known_pipeline_acc          : 78.4
  known_baseline_B            : 7.5
  known_pipeline_B            : 10.5
  known_pipeline_A_pct        : 42.1
  known_baseline_A_pct        : 93.4
  results_file                : /kaggle/working/piqa_ceiling/results.jsonl
  report_file                 : /kaggle/working/piqa_ceiling/eval_report.json
  angle1_file                 : /kaggle/working/piqa_ceiling/angle1_compute_efficiency.json
  angle2_file                 : /kaggle/working/piqa_ceiling/angle2_vote_consistency.json

In [5]:
# CELL 5 -- Load PIQA dataset
# Fields: goal, sol1, sol2, label (0=sol1 correct, 1=sol2 correct)
# We map label 0 -> 'A', label 1 -> 'B' for pipeline consistency.
import random

VALID_LETTERS  = set("AB")
LABEL_TO_LETTER = {0: "A", 1: "B"}

def normalise_piqa(item):
    q = (
        f"Goal: {item['goal'].strip()}\n\n"
        f"Which solution is more physically correct or practical?\n\n"
        f"A) {item['sol1'].strip()}\n"
        f"B) {item['sol2'].strip()}"
    )
    ans = LABEL_TO_LETTER[int(item["label"])]
    return {
        "question": q, "answer": ans,
        "goal": item["goal"], "sol1": item["sol1"], "sol2": item["sol2"],
    }

print("Loading PIQA from HuggingFace...")
raw_ds   = load_dataset(CONFIG["dataset_name"])
all_data = [normalise_piqa(x) for x in raw_ds[CONFIG["dataset_split"]]]

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Val size : {len(raw_ds[CONFIG['dataset_split']])}")
print(f"Formatted: {len(all_data)} questions")

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
n = min(CONFIG["max_eval_samples"], len(all_data))
test_data = random.sample(all_data, n)

a_count = sum(1 for x in test_data if x["answer"] == "A")
b_count = sum(1 for x in test_data if x["answer"] == "B")
print(f"Sampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
print(f"Label balance: A={a_count}  B={b_count}  (ideal: 50/50)")
print(f"\nSample question:\n{test_data[0]['question']}")
print(f"Answer: {test_data[0]['answer']}")

Loading PIQA from HuggingFace...


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.66M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/301k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16113 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3084 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1838 [00:00<?, ? examples/s]

Splits   : ['train', 'test', 'validation']
Val size : 1838
Formatted: 1838 questions
Sampled 500 questions (seed=42)
Label balance: A=249  B=251  (ideal: 50/50)

Sample question:
Goal: How do I choose good apples at the grocery store?

Which solution is more physically correct or practical?

A) Look for obvious bad spots. If you see spots that are rotten, dark brown, or too soft, the apple has likely already gone bad. ...    Look for cuts. ...    Examine the color. ...    Check the apple for firmness. ...    Sniff the apple to detect foul odor.
B) Look for obvious bad spots. If you see spots that are rotten, dark brown, or too soft, the apple has likely already gone bad. ...    Look for cuts cause those are the good ones. ...    Examine the color. ...    Check the apple for firmness. ...    Sniff the apple to detect foul odor.
Answer: A


In [6]:
# CELL 6 -- Answer extraction for binary choice (A or B)
# Tight extractor -- must not false-fire on mid-sentence A/B mentions.

def extract_gt_answer(answer_str):
    s = str(answer_str).strip().upper()
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    text = text.strip()
    # 1. Explicit conclusive phrases
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is|"
        r"best solution is|better solution is|solution is)"
        r"[\s:]*([AB])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 2. "Solution/Option A is correct/better/more practical"
    m = re.search(
        r"(?:solution|option)\s+([AB])\s+(?:is correct|is better|is more|is the|works better|is practical)",
        text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 3. #### A marker
    m = re.search(r"####\s*([AB])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 4. (A) or (B) at end
    m = re.search(r"\(([AB])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 5. **A** or **B**
    m = re.search(r"\*\*([AB])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 6. Standalone letter on its own line (last)
    matches = re.findall(r"^\s*([AB])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches: return matches[-1].upper()
    # 7. "I would choose / I choose"
    m = re.search(r"(?:i would choose|i choose|choose)\s+([AB])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 8. Last standalone A or B (loose fallback)
    matches = re.findall(r"\b([AB])\b", text, re.IGNORECASE)
    if matches: return matches[-1].upper()
    return ""

_tests = [
    ("The answer is A",              "A"),
    ("The best solution is B",       "B"),
    ("Solution A is more practical", "A"),
    ("#### B",                       "B"),
    ("(A)",                          "A"),
    ("**B**",                        "B"),
    ("I would choose A",             "A"),
    ("random text",                  ""),
]
ok = all(extract_pred_answer(t)==e for t,e in _tests)
print("Extractor:", "ALL PASSED" if ok else "FAILURES DETECTED")
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    print(f"  {'OK' if got==exp else 'FAIL'}  '{txt}' -> '{got}'")

Extractor: ALL PASSED
  OK  'The answer is A' -> 'A'
  OK  'The best solution is B' -> 'B'
  OK  'Solution A is more practical' -> 'A'
  OK  '#### B' -> 'B'
  OK  '(A)' -> 'A'
  OK  '**B**' -> 'B'
  OK  'I would choose A' -> 'A'
  OK  'random text' -> ''


In [7]:
# CELL 7 -- Load 3B model (BASE only -- NO LoRA, NO fine-tuning)
print(f"Loading: {CONFIG['model_name']}")
print("Adapter: NONE -- base model only, no task-specific training")

model_tok = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token

model_3b = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Model VRAM : {used:.2f} GB / {total:.1f} GB")
    print(f"Headroom   : {total - used:.1f} GB")

print(f"Compute per question: {CONFIG['model_params_B']}B x {CONFIG['n_votes']} = {CONFIG['model_params_B']*CONFIG['n_votes']}B param-passes")
print("Model ready -- no fine-tuning, no adapter")

Loading: Qwen/Qwen2.5-3B-Instruct
Adapter: NONE -- base model only, no task-specific training


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model VRAM : 5.75 GB / 15.9 GB
Headroom   : 10.1 GB
Compute per question: 3.0B x 5 = 15.0B param-passes
Model ready -- no fine-tuning, no adapter


In [8]:
# CELL 8 -- Generation functions
# PIQA-specific system prompts for the base 3B solver.

SOLVE_SYSTEM = (
    "You are a physical reasoning assistant.\n"
    "You are given a goal and two solutions (A and B).\n"
    "Decide which solution is more physically correct, practical, or effective.\n"
    "Think step by step if needed.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [A or B]"
)

REFINER_SYSTEM = (
    "You are a careful physical reasoning checker.\n"
    "You are given a goal with two solutions and a tie between A and B.\n"
    "Reason about which solution is more physically sound or practical.\n"
    "Your absolute last line must be exactly: The answer is [A or B]"
)

def run_3b(messages, max_tokens, temperature):
    prompt = model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = model_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    dev    = next(model_3b.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = model_3b.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = model_tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return model_tok.decode(new_toks, skip_special_tokens=True).strip()

def generate_solve(question):
    return run_3b(
        [{"role":"system","content":SOLVE_SYSTEM},
         {"role":"user",  "content":question}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )

def generate_refine(question, candidates):
    cands   = ", ".join(sorted(set(c for c in candidates if c)))
    content = f"{question}\n\nPrevious attempts gave tied answers: {cands}\nRe-reason carefully and pick A or B:"
    return run_3b(
        [{"role":"system","content":REFINER_SYSTEM},
         {"role":"user",  "content":content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )

print("Generation functions ready")
print(f"  generate_solve()   -- 3B base, temp {CONFIG['vote_temperature']}, no plan")
print(f"  generate_refine()  -- 3B base, temp {CONFIG['refiner_temperature']}, tie-breaker")

Generation functions ready
  generate_solve()   -- 3B base, temp 0.4, no plan
  generate_refine()  -- 3B base, temp 0.3, tie-breaker


In [9]:
# CELL 9 -- Voting logic (identical to pipeline notebook)
def vote_and_decide(answers, question, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid: valid = answers
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(answers)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / total
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])
    refiner_used = False; refiner_correct = None

    if is_majority:
        final = top_answer; strategy = "majority"
        conf  = round(top_count / total, 4); wasted = total - top_count
    else:
        ref_raw         = generate_refine(question, list(answers))
        ref_ans         = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None
        all_votes   = answers + [ref_ans]
        new_counts  = Counter(all_votes)
        new_common  = new_counts.most_common()
        new_top     = new_common[0][0]
        new_top_c   = new_common[0][1]
        still_tied  = len(new_common) > 1 and new_top_c == new_common[1][1]
        final       = new_top
        strategy    = "coin_flip" if still_tied else "refiner_tiebreak"
        conf        = round(new_top_c / len(all_votes), 4)
        vote_counts = new_counts; total = len(all_votes)
        correct_votes    = new_counts.get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / total
        wasted           = total - new_top_c

    return {
        "final_answer"    : final, "strategy":strategy, "confidence":conf,
        "vote_counts"     : dict(vote_counts), "correct_votes":correct_votes,
        "total_votes"     : total, "vote_consistency":round(vote_consistency,4),
        "wasted_votes"    : wasted, "refiner_used":refiner_used,
        "refiner_correct" : refiner_correct,
    }

print("Voting logic ready (majority / refiner_tiebreak / coin_flip)")

Voting logic ready (majority / refiner_tiebreak / coin_flip)


In [10]:
# CELL 10 -- Single question test
print("=" * 65)
print("SINGLE QUESTION TEST  (PIQA -- 3B Ceiling)")
print("=" * 65)
item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(q)
print(f"GT Answer: {gt}")
print(f"\nRunning {CONFIG['n_votes']} votes (3B base, no plan, temp={CONFIG['vote_temperature']})...")

votes_raw = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_solve(q)
    pred = extract_pred_answer(raw)
    votes_raw.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

dec = vote_and_decide(votes_raw, q, gt)
print(f"\n  Result    : {dec['final_answer']}  (GT: {gt})  {'CORRECT' if dec['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {dec['strategy']}")
print(f"  Confidence: {dec['confidence']}")
print(f"  Correct votes: {dec['correct_votes']}/{dec['total_votes']}")
print(f"  Vote counts  : {dec['vote_counts']}")
print("\nTest done -- run Cell 11 for full 500-question evaluation")

SINGLE QUESTION TEST  (PIQA -- 3B Ceiling)
Goal: How do I choose good apples at the grocery store?

Which solution is more physically correct or practical?

A) Look for obvious bad spots. If you see spots that are rotten, dark brown, or too soft, the apple has likely already gone bad. ...    Look for cuts. ...    Examine the color. ...    Check the apple for firmness. ...    Sniff the apple to detect foul odor.
B) Look for obvious bad spots. If you see spots that are rotten, dark brown, or too soft, the apple has likely already gone bad. ...    Look for cuts cause those are the good ones. ...    Examine the color. ...    Check the apple for firmness. ...    Sniff the apple to detect foul odor.
GT Answer: A

Running 5 votes (3B base, no plan, temp=0.4)...
  Vote 1: 'A'  |  raw[:80]: The answer is A
  Vote 2: 'A'  |  raw[:80]: The answer is A
  Vote 3: 'A'  |  raw[:80]: The answer is A
  Vote 4: 'A'  |  raw[:80]: The answer is A
  Vote 5: 'A'  |  raw[:80]: The answer is A

  Result    : 

In [11]:
# CELL 11 -- Full Evaluation Loop
print(f"PIQA 3B Ceiling evaluation: {len(test_data)} questions")
print(f"Model  : {CONFIG['model_name']} (base, no LoRA)")
print(f"Votes  : {CONFIG['n_votes']} x temp {CONFIG['vote_temperature']}")
print(f"Compute: {CONFIG['model_params_B']}B x {CONFIG['n_votes']} = {CONFIG['model_params_B']*CONFIG['n_votes']}B param-passes")
print(f"Random chance: {CONFIG['random_chance']}% (binary A/B)")
print("-" * 65)

results   = []
start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from index {start_idx} ({len(results)} saved)")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="PIQA 3B Ceiling"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    try:
        votes_raw = [extract_pred_answer(generate_solve(question))
                     for _ in range(CONFIG["n_votes"])]
        dec = vote_and_decide(votes_raw, question, gt_answer)
        results.append({
            "mode"            : "ceiling_3b",
            "idx"             : idx,
            "question"        : question,
            "gt_answer"       : gt_answer,
            "final_answer"    : dec["final_answer"],
            "correct"         : dec["final_answer"] == gt_answer,
            "strategy"        : dec["strategy"],
            "confidence"      : dec["confidence"],
            "correct_votes"   : dec["correct_votes"],
            "total_votes"     : dec["total_votes"],
            "vote_consistency": dec["vote_consistency"],
            "wasted_votes"    : dec["wasted_votes"],
            "refiner_used"    : dec["refiner_used"],
            "refiner_correct" : dec["refiner_correct"],
            "vote_counts"     : dec["vote_counts"],
        })
    except RuntimeError as e:
        results.append({
            "mode":"ceiling_3b","idx":idx,"question":question,
            "gt_answer":gt_answer,"final_answer":"","correct":False,"strategy":"error",
            "confidence":0.0,"correct_votes":0,"total_votes":CONFIG["n_votes"],
            "vote_consistency":0.0,"wasted_votes":CONFIG["n_votes"],
            "refiner_used":False,"refiner_correct":None,"vote_counts":{},"error":str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"],"w") as f:
            for r in results: f.write(json.dumps(r)+"\n")
        with open(CONFIG["checkpoint_file"],"w") as f:
            json.dump({"last_index":idx+1},f)
        acc  = sum(r["correct"] for r in results)/len(results)*100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:3d}/{len(test_data)}]  3B Ceiling: {acc:.1f}%  ({mins:.1f} min)")

with open(CONFIG["results_file"],"w") as f:
    for r in results: f.write(json.dumps(r)+"\n")
with open(CONFIG["checkpoint_file"],"w") as f:
    json.dump({"last_index":len(test_data)},f)

correct = sum(r["correct"] for r in results)
acc     = correct/len(results)*100
print(f"\nEvaluation complete.")
print(f"  3B Ceiling : {correct}/{len(results)} = {acc:.1f}%")
print(f"  Pipeline   : {CONFIG['known_pipeline_acc']}%  (known, 10.5B, N=500)")
print(f"  Baseline   : {CONFIG['known_baseline_acc']}%  (known, 7.5B, N=500)")

PIQA 3B Ceiling evaluation: 500 questions
Model  : Qwen/Qwen2.5-3B-Instruct (base, no LoRA)
Votes  : 5 x temp 0.4
Compute: 3.0B x 5 = 15.0B param-passes
Random chance: 50.0% (binary A/B)
-----------------------------------------------------------------
Starting fresh


PIQA 3B Ceiling:   0%|          | 0/500 [00:00<?, ?it/s]

  [ 25/500]  3B Ceiling: 72.0%  (4.9 min)
  [ 50/500]  3B Ceiling: 68.0%  (11.1 min)
  [ 75/500]  3B Ceiling: 72.0%  (16.2 min)
  [100/500]  3B Ceiling: 76.0%  (20.9 min)
  [125/500]  3B Ceiling: 72.8%  (26.8 min)
  [150/500]  3B Ceiling: 74.0%  (32.9 min)
  [175/500]  3B Ceiling: 73.7%  (39.0 min)
  [200/500]  3B Ceiling: 71.0%  (44.5 min)
  [225/500]  3B Ceiling: 71.1%  (51.7 min)
  [250/500]  3B Ceiling: 71.6%  (57.4 min)
  [275/500]  3B Ceiling: 72.7%  (62.9 min)
  [300/500]  3B Ceiling: 73.7%  (70.7 min)
  [325/500]  3B Ceiling: 75.1%  (77.1 min)
  [350/500]  3B Ceiling: 74.9%  (80.9 min)
  [375/500]  3B Ceiling: 74.7%  (84.7 min)
  [400/500]  3B Ceiling: 74.5%  (90.0 min)
  [425/500]  3B Ceiling: 74.6%  (96.8 min)
  [450/500]  3B Ceiling: 75.1%  (100.8 min)
  [475/500]  3B Ceiling: 74.7%  (105.3 min)
  [500/500]  3B Ceiling: 75.2%  (112.8 min)

Evaluation complete.
  3B Ceiling : 376/500 = 75.2%
  Pipeline   : 78.4%  (known, 10.5B, N=500)
  Baseline   : 54.2%  (known, 7.5B, N=500

In [12]:
# CELL 12 -- ANGLE 1: COMPUTE EFFICIENCY
ceiling_compute  = CONFIG["model_params_B"] * CONFIG["n_votes"]
pipeline_compute = CONFIG["known_pipeline_B"]
baseline_compute = CONFIG["known_baseline_B"]

ceiling_acc  = sum(r["correct"] for r in results)/len(results)*100
pipeline_acc = CONFIG["known_pipeline_acc"]
baseline_acc = CONFIG["known_baseline_acc"]
random_chance = CONFIG["random_chance"]

ceiling_eff  = ceiling_acc  / ceiling_compute
pipeline_eff = pipeline_acc / pipeline_compute
baseline_eff = baseline_acc / baseline_compute

n = len(results); N = CONFIG["n_votes"]
ceiling_wasted = sum(r["wasted_votes"] for r in results)
strategy_stats = {}
for r in results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

ref_triggered = sum(r["refiner_used"] for r in results)
ref_correct   = sum(1 for r in results if r["refiner_used"] and r.get("refiner_correct"))

print("=" * 68)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (PIQA -- Three-Way Comparison)")
print("=" * 68)
print(f"  Random chance baseline: {random_chance}% (binary A/B choice)")
print()
print(f"  {'Setup':<38} | {'Compute':>8} | {'Accuracy':>9} | {'Above Chance':>13} | {'Acc/B':>7}")
print(f"  {'-'*38}-+-{'-'*8}-+-{'-'*9}-+-{'-'*13}-+-{'-'*7}")
print(f"  {'Baseline  (1.5B x 5)':<38} | {baseline_compute:>6.1f}B  | {baseline_acc:>8.1f}% | {baseline_acc-random_chance:>+12.1f}% | {baseline_eff:>6.3f}")
print(f"  {'Pipeline  (3B+LoRA x1 + 1.5B x5)':<38} | {pipeline_compute:>6.1f}B  | {pipeline_acc:>8.1f}% | {pipeline_acc-random_chance:>+12.1f}% | {pipeline_eff:>6.3f}")
print(f"  {'Ceiling   (3B base x 5)  <- THIS':<38} | {ceiling_compute:>6.1f}B  | {ceiling_acc:>8.1f}% | {ceiling_acc-random_chance:>+12.1f}% | {ceiling_eff:>6.3f}")
print()
print(f"  Pipeline vs Ceiling  : {pipeline_acc - ceiling_acc:+.1f} pts  (pipeline uses {ceiling_compute - pipeline_compute:.1f}B LESS)")
print(f"  Ceiling  vs Baseline : {ceiling_acc  - baseline_acc:+.1f} pts  (+{ceiling_compute - baseline_compute:.1f}B more)")
print(f"  Pipeline vs Baseline : {pipeline_acc - baseline_acc:+.1f} pts  (+{pipeline_compute - baseline_compute:.1f}B more)")
print()
if pipeline_acc >= ceiling_acc:
    print(f"  RESULT: Pipeline MATCHES or BEATS ceiling at 30% lower cost.")
    print(f"  Fine-tuned planning + bias correction > raw model capacity scaling on PIQA.")
else:
    gap  = ceiling_acc - pipeline_acc
    frac = (pipeline_acc - baseline_acc) / max(ceiling_acc - baseline_acc, 0.01) * 100
    print(f"  RESULT: 3B ceiling leads pipeline by {gap:.1f} pts at 43% higher cost.")
    print(f"  Pipeline delivers {frac:.0f}% of ceiling gain at {pipeline_compute/ceiling_compute*100:.0f}% of ceiling cost.")

print(f"\n  Wasted votes -- Ceiling: {ceiling_wasted}/{n*N} ({ceiling_wasted/(n*N)*100:.1f}%)")
if ref_triggered > 0:
    print(f"  Refiner: triggered {ref_triggered}, correct {ref_correct}")
print(f"\n  Strategy breakdown:")
for s,v in sorted(strategy_stats.items(), key=lambda x:-x[1]['n']):
    acc_s = v['correct']/v['n']*100 if v['n'] else 0
    print(f"    {s:<22}  n={v['n']:3d}  acc={acc_s:.1f}%")

angle1 = {
    "dataset":"PIQA","experiment":"ceiling_3b","n_questions":n,
    "random_chance":random_chance,
    "ceiling_accuracy":round(ceiling_acc,2),"pipeline_accuracy":pipeline_acc,"baseline_accuracy":baseline_acc,
    "ceiling_compute_B":ceiling_compute,"pipeline_compute_B":pipeline_compute,"baseline_compute_B":baseline_compute,
    "ceiling_efficiency":round(ceiling_eff,4),"pipeline_efficiency":round(pipeline_eff,4),"baseline_efficiency":round(baseline_eff,4),
    "ceiling_wasted_votes":ceiling_wasted,"refiner_triggered":ref_triggered,"refiner_correct":ref_correct,
    "strategy_breakdown":strategy_stats,"pipeline_beats_ceiling":pipeline_acc >= ceiling_acc,
}
with open(CONFIG["angle1_file"],"w") as f: json.dump(angle1,f,indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")

ANGLE 1 -- COMPUTE EFFICIENCY  (PIQA -- Three-Way Comparison)
  Random chance baseline: 50.0% (binary A/B choice)

  Setup                                  |  Compute |  Accuracy |  Above Chance |   Acc/B
  ---------------------------------------+----------+-----------+---------------+--------
  Baseline  (1.5B x 5)                   |    7.5B  |     54.2% |         +4.2% |  7.227
  Pipeline  (3B+LoRA x1 + 1.5B x5)       |   10.5B  |     78.4% |        +28.4% |  7.467
  Ceiling   (3B base x 5)  <- THIS       |   15.0B  |     75.2% |        +25.2% |  5.013

  Pipeline vs Ceiling  : +3.2 pts  (pipeline uses 4.5B LESS)
  Ceiling  vs Baseline : +21.0 pts  (+7.5B more)
  Pipeline vs Baseline : +24.2 pts  (+3.0B more)

  RESULT: Pipeline MATCHES or BEATS ceiling at 30% lower cost.
  Fine-tuned planning + bias correction > raw model capacity scaling on PIQA.

  Wasted votes -- Ceiling: 109/2500 (4.4%)

  Strategy breakdown:
    majority                n=500  acc=75.2%

Saved -> /kaggle/workin

In [13]:
# CELL 13 -- ANGLE 2: VOTE CONSISTENCY + A/B POSITION BIAS
# ─────────────────────────────────────────────────────────────────────
# PIQA KEY FINDING: the 1.5B baseline picks A on 93.4% of questions.
# The pipeline's fine-tuned guide corrected this to 42.1%.
# Does the 3B BASE model suffer the same A-bias, or does scale help?
# ─────────────────────────────────────────────────────────────────────

cons_scores = [r["vote_consistency"] for r in results]
mean_cons   = np.mean(cons_scores)

def bucket(scores):
    return {
        "all_wrong  (0%)"  : sum(1 for s in scores if s == 0.0),
        "low       (1-39%)": sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)" : sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)" : sum(1 for s in scores if s >= 0.8),
    }

dist      = bucket(cons_scores)
corr_cons = [r["vote_consistency"] for r in results if r["correct"]]

# Position bias: track A vs B vote distribution
a_votes = 0; b_votes = 0; total_votes = 0
for r in results:
    for letter, count in r["vote_counts"].items():
        if letter == "A": a_votes += count
        elif letter == "B": b_votes += count
        if letter in VALID_LETTERS: total_votes += count

a_pct = a_votes / max(total_votes, 1) * 100
b_pct = b_votes / max(total_votes, 1) * 100

# Known pipeline/baseline values
pipeline_cons = 0.778   # 77.8% from PIQA N=500
baseline_cons = 0.544   # 54.4% from PIQA N=500
pipeline_dist = {"all_wrong  (0%)":100,"low       (1-39%)":3,"medium  (40-79%)":16,"high   (80-100%)":381}

print("=" * 68)
print("ANGLE 2 -- VOTE CONSISTENCY + POSITION BIAS  (PIQA -- Three-Way)")
print("=" * 68)
print(f"  Mean correct-vote ratio (out of {CONFIG['n_votes']} per question):")
print(f"    Baseline (1.5B x5) : {baseline_cons*100:.1f}%  ({baseline_cons*5:.2f}/5 avg)")
print(f"    Pipeline (3B+LoRA) : {pipeline_cons*100:.1f}%  ({pipeline_cons*5:.2f}/5 avg)")
print(f"    Ceiling  (3B base) : {mean_cons*100:.1f}%  ({mean_cons*5:.2f}/5 avg)  <- THIS")

print(f"\n  Distribution (3B Ceiling vs Pipeline):")
print(f"  {'Bucket':<22} | {'Ceiling':>8} | {'Pipeline':>8}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}")
for bkt in ["all_wrong  (0%)","low       (1-39%)","medium  (40-79%)","high   (80-100%)"]:
    cv = dist[bkt]; pv = pipeline_dist.get(bkt,"—")
    print(f"  {bkt:<22} | {cv:>8} | {pv:>8}")

print(f"\n  ── A/B POSITION BIAS ──────────────────────────────────────────")
print(f"  Expected: 50% A, 50% B (uniform for binary choice)")
print()
print(f"  {'Condition':<32} | {'A %':>7} | {'B %':>7} | {'Bias?':>20}")
print(f"  {'-'*32}-+-{'-'*7}-+-{'-'*7}-+-{'-'*20}")
print(f"  {'Baseline (1.5B x5)':<32} | {CONFIG['known_baseline_A_pct']:>6.1f}% | {100-CONFIG['known_baseline_A_pct']:>6.1f}% | {'SEVERE A-collapse ⚠' if CONFIG['known_baseline_A_pct']>70 else 'OK':>20}")
print(f"  {'Pipeline (3B+LoRA)':<32} | {CONFIG['known_pipeline_A_pct']:>6.1f}% | {100-CONFIG['known_pipeline_A_pct']:>6.1f}% | {'Corrected by guide':>20}")
print(f"  {'Ceiling  (3B base) <- THIS':<32} | {a_pct:>6.1f}% | {b_pct:>6.1f}% | ", end="")

if a_pct > 70:
    print(f"{'SEVERE A-collapse ⚠':>20}")
    print(f"\n  ⚠ 3B base model ALSO has A-bias ({a_pct:.1f}%). Position bias is model-size independent.")
    print(f"  Fine-tuning (not scale) is what suppresses positional heuristics.")
elif a_pct > 55:
    print(f"{'Mild A-preference':>20}")
    print(f"\n  Mild A-preference ({a_pct:.1f}%). 3B base partially suppresses bias vs 1.5B baseline.")
else:
    print(f"{'Near-uniform ✓':>20}")
    print(f"\n  Near-uniform ({a_pct:.1f}% A). 3B base model suppresses A-bias through scale alone.")

if corr_cons:
    print(f"\n  Correct questions: ceiling consistency = {np.mean(corr_cons)*100:.1f}% (n={len(corr_cons)})")

angle2 = {
    "dataset":"PIQA","experiment":"ceiling_3b","n_questions":len(results),
    "ceiling_mean_consistency":round(mean_cons,4),
    "pipeline_mean_consistency":pipeline_cons,"baseline_mean_consistency":baseline_cons,
    "ceiling_distribution":dist,
    "ceiling_a_pct":round(a_pct,2),"ceiling_b_pct":round(b_pct,2),
    "pipeline_a_pct":CONFIG["known_pipeline_A_pct"],"baseline_a_pct":CONFIG["known_baseline_A_pct"],
    "ceiling_correct_q_consistency":round(np.mean(corr_cons),4) if corr_cons else 0,
}
with open(CONFIG["angle2_file"],"w") as f: json.dump(angle2,f,indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")

ANGLE 2 -- VOTE CONSISTENCY + POSITION BIAS  (PIQA -- Three-Way)
  Mean correct-vote ratio (out of 5 per question):
    Baseline (1.5B x5) : 54.4%  (2.72/5 avg)
    Pipeline (3B+LoRA) : 77.8%  (3.89/5 avg)
    Ceiling  (3B base) : 74.2%  (3.71/5 avg)  <- THIS

  Distribution (3B Ceiling vs Pipeline):
  Bucket                 |  Ceiling | Pipeline
  -----------------------+----------+---------
  all_wrong  (0%)        |       97 |      100
  low       (1-39%)      |       12 |        3
  medium  (40-79%)       |       37 |       16
  high   (80-100%)       |      354 |      381

  ── A/B POSITION BIAS ──────────────────────────────────────────
  Expected: 50% A, 50% B (uniform for binary choice)

  Condition                        |     A % |     B % |                Bias?
  ---------------------------------+---------+---------+---------------------
  Baseline (1.5B x5)               |   93.4% |    6.6% |  SEVERE A-collapse ⚠
  Pipeline (3B+LoRA)               |   42.1% |   57.9% |   Co

In [14]:
# CELL 14 -- ANGLE 3: CONFIDENCE CALIBRATION
def calibration_report(res, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40,  0.25),
    ]
    n_total = len(res); ece = 0.0; calib_out = []
    false_conf = sum(1 for r in res if r["confidence"] >= 0.80 and not r["correct"])
    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in res if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |"); continue
        n=len(subset); acc=sum(r["correct"] for r in subset)/n; gap=abs(acc-mid)
        ece += (n/n_total)*gap; flag="Good" if gap<0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),"expected":mid,"gap":round(gap,4)})
    hc = [r for r in res if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc)/max(1,len(hc))*100
    print(f"  {'ECE (lower=better)':<26}   {ece:.4f}")
    print(f"  High-conf questions : {len(hc)}  |  Accuracy when confident: {hc_acc:.1f}%")
    print(f"  Confidently WRONG   : {false_conf}")
    return ece, calib_out, false_conf

# Known ECE from N=500 pipeline run
known_pipe_ece = 0.1096
known_base_ece = 0.3544

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (PIQA -- 3B Ceiling)")
print("=" * 65)
print("Note: Baseline ECE=0.354 was catastrophic -- A-bias made the model")
print("uniformly confident (100%) on ~491 questions with only 54.8% accuracy.")
ceiling_ece, ceiling_calib, ceiling_false = calibration_report(results, "3B CEILING (this run)")
print(f"\n  ECE -- Three-Way:")
print(f"    Baseline (1.5B x5) : {known_base_ece:.4f}  ← catastrophic A-bias ECE")
print(f"    Pipeline (3B+LoRA) : {known_pipe_ece:.4f}  ← bias corrected by guide")
print(f"    Ceiling  (3B base) : {ceiling_ece:.4f}  <- THIS")
print(f"\n  Ceiling vs Pipeline: {ceiling_ece - known_pipe_ece:+.4f} ({'worse' if ceiling_ece > known_pipe_ece else 'better'})")
print(f"  Ceiling vs Baseline: {ceiling_ece - known_base_ece:+.4f} ({'worse' if ceiling_ece > known_base_ece else 'better'})")
print(f"  False confidence (ceiling): {ceiling_false}")
print(f"  (Baseline false confidence was 222 -- due to A-bias unanimous wrong answers)")

angle3 = {
    "dataset":"PIQA","experiment":"ceiling_3b","n_questions":len(results),
    "ceiling_ece":round(ceiling_ece,4),"pipeline_ece":known_pipe_ece,"baseline_ece":known_base_ece,
    "ceiling_false_confidence":ceiling_false,"ceiling_calibration":ceiling_calib,
}
with open(CONFIG["angle3_file"],"w") as f: json.dump(angle3,f,indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")

ANGLE 3 -- CONFIDENCE CALIBRATION  (PIQA -- 3B Ceiling)
Note: Baseline ECE=0.354 was catastrophic -- A-bias made the model
uniformly confident (100%) on ~491 questions with only 54.8% accuracy.

  [3B CEILING (this run)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   463 |     76.5% |       90% |  0.135 | Good
  High       (0.60-0.80)     |    37 |     59.5% |       70% |  0.105 | Good
  Medium     (0.40-0.60)     |    -- |        -- |       50% |     -- |
  Low        (<0.40)         |    -- |        -- |       25% |     -- |
  ECE (lower=better)           0.1332
  High-conf questions : 463  |  Accuracy when confident: 76.5%
  Confidently WRONG   : 109

  ECE -- Three-Way:
    Baseline (1.5B x5) : 0.3544  ← catastrophic A-bias ECE
    Pipeline (3B+LoRA) : 0.1096  ← bias corrected by guide
    Ceiling  (3B base) : 0.1332  <- THIS

  Ceiling vs Pipel

In [15]:
# CELL 15 -- Full Three-Way Summary Table
with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

ca = a1["ceiling_accuracy"]
pa = a1["pipeline_accuracy"]
ba = a1["baseline_accuracy"]

print("=" * 75)
print("  PIQA -- THREE-WAY: BASELINE / PIPELINE / 3B CEILING")
print(f"  N={a1['n_questions']} questions  |  Seed={CONFIG['random_seed']}  |  Qwen2.5 model family")
print(f"  Random chance: {CONFIG['random_chance']}% (binary A/B)")
print("=" * 75)

rows = [
    ["Metric",                  "Baseline",          "Our Pipeline",          "3B Ceiling (this)"],
    ["Model",                   "Qwen2.5-1.5B x5",   "3B LoRA + 1.5B x5",    "Qwen2.5-3B base x5"],
    ["Fine-Tuning",             "None",              "LoRA on GSM8K",         "None"],
    ["Compute (param-passes)",  "7.5B",              "10.5B",                 "15.0B"],
    ["─────────────────────",   "───────────────",   "───────────────────",   "──────────────────"],
    ["Overall Accuracy",        f"{ba:.1f}%",         f"{pa:.1f}%",            f"{ca:.1f}%"],
    ["Above Random (50%)",      f"+{ba-50:.1f} pts",  f"+{pa-50:.1f} pts",     f"+{ca-50:.1f} pts"],
    ["vs Baseline",             "—",                 f"+{pa-ba:.1f} pts",      f"+{ca-ba:.1f} pts"],
    ["Pipeline vs Ceiling",     "—",                 f"{'WINS' if pa>=ca else 'loses'} {abs(pa-ca):.1f} pts","←"],
    ["Acc / Billion passes",    f"{ba/7.5:.3f}",      f"{pa/10.5:.3f}",        f"{ca/15.0:.3f}"],
    ["─────────────────────",   "───────────────",   "───────────────────",   "──────────────────"],
    ["Vote Consistency",        f"{a2['baseline_mean_consistency']*100:.1f}%",
                                 f"{a2['pipeline_mean_consistency']*100:.1f}%",
                                 f"{a2['ceiling_mean_consistency']*100:.1f}%"],
    ["A-option bias",           f"{a2['baseline_a_pct']:.1f}% A-votes",
                                 f"{a2['pipeline_a_pct']:.1f}% A-votes",
                                 f"{a2['ceiling_a_pct']:.1f}% A-votes"],
    ["ECE (lower=better)",      f"{a3['baseline_ece']:.4f}",
                                 f"{a3['pipeline_ece']:.4f}",
                                 f"{a3['ceiling_ece']:.4f}"],
    ["False Confidence",        "222 (A-bias)",       "103",                   str(a3["ceiling_false_confidence"])],
]

col_w = [26, 20, 22, 20]
sep   = "-+-".join("-"*w for w in col_w)
for i, row in enumerate(rows):
    if "─────" in row[0]:
        print("  " + sep); continue
    line = " | ".join(str(c).ljust(col_w[j]) for j,c in enumerate(row))
    print("  " + line)
    if i == 0: print("  " + sep)

print()
print("  ── A/B Bias Story ────────────────────────────────────────────────")
print(f"  Baseline:  {a2['baseline_a_pct']:.1f}% A  (catastrophic collapse -- expected 50%)")
print(f"  Pipeline:  {a2['pipeline_a_pct']:.1f}% A  (guide corrected bias -- near-uniform)")
print(f"  Ceiling:   {a2['ceiling_a_pct']:.1f}% A  <- what does the 3B base do?")
if a2['ceiling_a_pct'] > 70:
    print(f"  → 3B base ALSO has A-bias. Positional heuristics persist at 3B scale.")
    print(f"  → Fine-tuning, not model size, is what suppresses position bias.")
elif a2['ceiling_a_pct'] > 55:
    print(f"  → 3B base has partial A-preference but much weaker than 1.5B baseline.")
    print(f"  → Scale partially helps; fine-tuned guidance is still more effective.")
else:
    print(f"  → 3B base is near-uniform. Scale alone suppresses positional heuristics.")

print()
print("=" * 75)
if pa >= ca:
    print(f"  VERDICT: Pipeline BEATS the 3B ceiling (+{pa-ca:.1f} pts) at 30% lower cost.")
    print(f"  Structured planning + bias correction is more efficient than raw scale.")
else:
    gap  = ca - pa
    frac = (pa - ba) / max(ca - ba, 0.01) * 100
    print(f"  VERDICT: 3B ceiling leads pipeline by {gap:.1f} pts.")
    print(f"  Pipeline delivers {frac:.0f}% of ceiling gain at {pa/ca*100:.0f}% of ceiling cost.")
print("=" * 75)

full = {
    "dataset":"PIQA","seed":CONFIG["random_seed"],"n_questions":a1["n_questions"],
    "conditions":{
        "baseline":{"compute_B":7.5, "accuracy":ba,"model":"Qwen2.5-1.5B x5","fine_tuned":False},
        "pipeline":{"compute_B":10.5,"accuracy":pa,"model":"3B LoRA + 1.5B x5","fine_tuned":True},
        "ceiling" :{"compute_B":15.0,"accuracy":ca,"model":"3B base x5","fine_tuned":False},
    },
    "pipeline_beats_ceiling":pa >= ca,
    "angle1":a1,"angle2":a2,"angle3":a3,
}
with open(CONFIG["report_file"],"w") as f: json.dump(full,f,indent=2)
print(f"\nAll results saved to {OUTPUT_DIR}/")
print("Files: results.jsonl · eval_report.json · angle1/2/3.json")

  PIQA -- THREE-WAY: BASELINE / PIPELINE / 3B CEILING
  N=500 questions  |  Seed=42  |  Qwen2.5 model family
  Random chance: 50.0% (binary A/B)
  Metric                     | Baseline             | Our Pipeline           | 3B Ceiling (this)   
  ---------------------------+----------------------+------------------------+---------------------
  Model                      | Qwen2.5-1.5B x5      | 3B LoRA + 1.5B x5      | Qwen2.5-3B base x5  
  Fine-Tuning                | None                 | LoRA on GSM8K          | None                
  Compute (param-passes)     | 7.5B                 | 10.5B                  | 15.0B               
  ---------------------------+----------------------+------------------------+---------------------
  Overall Accuracy           | 54.2%                | 78.4%                  | 75.2%               
  Above Random (50%)         | +4.2 pts             | +28.4 pts              | +25.2 pts           
  vs Baseline                | —                    | +